In [2]:
import pandas as pd
from dotenv import load_dotenv
import os
load_dotenv()

True

In [ ]:
# populate db with old db

In [3]:
def get_technician_dict():
    """Cette fonction sers à se connecter à une BDD pour récuperer les informations d'anonymisation de techniciens"""
    import mysql.connector as bdd_connect
    BDD_CW = bdd_connect.connect(host='localhost',
                                user=os.getenv('LOCAL_DB_USER'),
                                password=os.getenv('LOCAL_DB_PASS'),
                                database=os.getenv('CW_DB_NAME'),
                                port=3306)
    curseur = BDD_CW.cursor()
    # Query avec docstring pour eviter SQL insertion
    to_execute = f"SELECT Technicien_id, Initiales_tech FROM Techniciens"
    curseur.execute(to_execute)
    # Fetch data 
    rows=curseur.fetchall()
    curseur.close()
    BDD_CW.close()
    # Export
    df = dict(pd.DataFrame(rows, columns=[i[0] for i in curseur.description]).values)
    return df

In [9]:
data = pd.read_csv("API_ML/ML_models/ML_supervized/data/2025-03-14_data_production.csv")
print("Dimensions données source :",  data.shape)

# Select given columns
new_data = data[['Batch_name','Date','Technicien_id','Batch_XX','Masse_XX','Batch_YY_ref','Volume_YY',
    'BaG_T','Vitesse_agitation',
    'Heure_debut','Heure_ajout2','Lab_HR','Lab_T','QC_Conc_XY','QC_Categorie']]

new_data.columns = ['Batch_XY_name','Batch_XY_date','Batch_XY_Technicien','Batch_XY_XX_batch',
                'Batch_XY_XX_masse','Batch_XY_YY_batch','Batch_XY_YY_Volume','Batch_XY_Temperature',
                'Batch_XY_Agitation','Batch_XY_heure_debut','Batch_XY_heure_fin',
                'Batch_XY_room_HR','Batch_XY_room_T','Batch_XY_Stock','Batch_XY_Analyses']
new_data.loc[:,'Batch_XY_Stock'] = 0

# Get id of technicians
df = get_technician_dict()
# df = {str(k):v for k,v in df.items()}

# Readjust columns types
new_data.loc[:,'Batch_XY_Technicien'] = new_data['Batch_XY_Technicien'].apply(lambda x : df[(x)]).astype(str).tolist()
new_data.loc[:,'Batch_XY_Analyses'] = data.Centrifugation.astype(str).tolist()
new_data.loc[:,'Batch_XY_YY_batch'] = 'to_fill'

# export to new database
import json
import requests

headers = {
'accept': 'application/json',
'Content-Type': 'application/json'}

for i in range(new_data.shape[0]):
    to_bdd = new_data.iloc[i,:].fillna(0).to_dict()

    response = requests.post('http://127.0.0.1:8000/XY/', headers=headers, data=json.dumps(to_bdd))



    ### ADD PRECITIONS DATA TOO
    response = requests.post(url = "http://127.0.0.1:8001/predict/elasticnet",
                                headers = {'Supervized-API-Key': os.getenv('API_SUPERVIZED_SECRET_KEY'),
                                           'Content-Type': 'application/json'},
                                data = json.dumps(to_bdd))

    pred = response.json()['pred']

    if response.status_code != 200:
        print("ERROR", response.content)

Dimensions données source : (280, 55)


/var/folders/dv/gzhyqctn53s9bh23g7tbvl940000gn/T/ipykernel_9385/4032705593.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['IT', 'IT', 'IT', 'CD', 'IT', 'IT', 'CD', 'IT', 'CD', 'IT', 'IT', 'CD', 'IT', 'RS', 'CD', 'IT', 'IT', 'CD', 'IT', 'IT', 'CD', 'IT', 'IT', 'CD', 'IT', 'IT', 'IT', 'CD', 'IT', 'MM', 'IT', 'IT', 'FB', 'IT', 'FB', 'IT', 'IT', 'MM', 'CD', 'IT', 'CD', 'JP', 'CD', 'JP', 'IT', 'JP', 'JP', 'IT', 'JP', 'JP', 'CD', 'JP', 'JP', 'CD', 'JP', 'JP', 'JP', 'JP', 'CD', 'JP', 'JP', 'CD', 'JP', 'CD', 'JP', 'CD', 'JP', 'JP', 'CD', 'JP', 'JP', 'JP', 'JP', 'CD', 'JP', 'CD', 'CD', 'JP', 'CD', 'CD', 'JP', 'CD', 'CD', 'JP', 'JP', 'CD', 'JP', 'CD', 'JP', 'JP', 'CD', 'JP', 'JP', 'CD', 'JP', 'CD', 'JP', 'JP', 'JP', 'JP', 'JP', 'JP', 'JP', 'JP', 'JP', 'CD', 'CD', 'JP', 'JP', 'NM', 'JP', 'NM', 'CD', 'JP', 'NM', 'NM', 'JP', 'NM', 'NM', 'JP', 'CD', 'NM', 'NM', 'CD', 'JP', 'MM', 'CD', 'CD', 'NM', 'NM', 'NM', 'CD', 'NM', '

In [10]:
# populate BDD with TRUE UV analysis
import numpy as np
import requests
import json
headers={'accept':'application/json',
         'Content-type':'application/json'}
not_found = list()
for i in range(UV_data.shape[0]):
    try:
        # assert UV_data.iloc[i,:].name.split("/")[-1] in batch_dilution.keys()
        # get centrif
        centrif = 1 if 'NC' in UV_data.iloc[i,:].name.split("/")[-1].upper() else 0
        # get ratio
        index = np.argmax(UV_data.iloc[i,50:100]) # index 50:100 correspond au wavelength 250:300
        lambda_max = UV_data.iloc[i,:].index[index+50]
        abs_lambda_max = UV_data.iloc[i,index+50]
        abs_800 = UV_data.iloc[i,-1]
        ratio = UV_data.iloc[i,index+50] / UV_data.iloc[i,-1]
        dilution = 10
        # prepare data
        data={'Analyse_name': UV_data.iloc[i,:].name.split("/")[-1].replace(' ','')[:5],
              'Analyse_subname':'UV',
              'Analyse_details':{'Dilution':f"1:{dilution}",
                                 'Centrifuge':centrif,
                                 'ratio':ratio,
                                 'conc':abs_800*dilution},
              'Analyses_data':UV_data.iloc[i,:].to_dict()}
        
        response = requests.post('http://127.0.0.1:8000/analyses/',headers=headers,data=json.dumps(data))
    

        ############################################
        #### MISE A JOUR PREDICTION WITH REAL DATA
        response = requests.get("http://127.0.0.1:8000/Predictions")
        code_get = response.status_code

        # update the prediction for a given batch name
        a = pd.DataFrame(response.json())
        batch_name = data['Analyse_name']
        mask = [batch_name in a.Prediction_data[i]['sample'] for i in range(a.shape[0])]
        new_pred=dict()
        for col in a[mask]:
            new_pred[col] = a[mask][col].values[0]
        id = new_pred['Prediction_id']
        new_pred['Prediction_id'] = str(new_pred['Prediction_id'])

        # mise a jour concentration reelle
        new_pred_data = new_pred['Prediction_data']
        new_pred_data['reel'] = abs_800*dilution
        new_pred['Prediction_data'] = new_pred_data

        response = requests.post(f"http://127.0.0.1:8000/Predictions/update/{id}",data=json.dumps(new_pred))
        code_post = response.status_code
        print(code_get, code_post)
    
    
    except Exception as e :
        print(e)
        not_found.append(UV_data.iloc[i,:].name.split("/")[-1])
len(not_found)

200 200
200 200
200 200
200 200
200 200
200 200
200 200
index 0 is out of bounds for axis 0 with size 0
index 0 is out of bounds for axis 0 with size 0
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
index 0 is out of bounds for axis 0 with size 0
200 200
200 200
200 200
200 200
200 200
200 200
200 200
index 0 is out of bounds for axis 0 with size 0
index 0 is out of bounds for axis 0 with size 0
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
index 0 is out of bounds for axis 0 with size 0
index 0 is out of bounds for axis 0 with size 0
index 0 is out of bounds for axis 0 with size 0
index 0 is out of bounds for axis 0 with size 0
200 200
200 200
200 200
200 200
200 200
200 200
200 200
200 200
index 0 is out of bounds for axis 0 with size 0
200 200
200 200
index 0 is out of bounds for axis 0 with size 0
index 0 is out of bounds for axis 0 with size 0
200 200
200 200
200 200
200 200


107

In [5]:
response = requests.get('http://127.0.0.1:8000/XY/')

c = pd.DataFrame(response.json())
c

,Batch_OGD_id,Batch_OGD_name,Batch_OGD_date,Batch_OGD_Technicien,Batch_OGD_KC8_batch,Batch_OGD_KC8_masse,Batch_OGD_THF_batch,Batch_OGD_THF_Volume,Batch_OGD_Temperature,Batch_OGD_Agitation,Batch_OGD_heure_debut,Batch_OGD_heure_fin,Batch_OGD_room_HR,Batch_OGD_room_T,Batch_OGD_Stock,Batch_OGD_Analyses
0,1,2301A,2023-01-04T00:00:00,IT,K01,5.0000,to_fill,500,0.0,230,2023-01-04T11:00:00,2023-01-10T00:00:00,34.3,20.0,0.0,1
1,2,2301B,2023-01-05T00:00:00,IT,K01,5.0000,to_fill,500,0.0,230,2023-01-05T12:10:00,2023-01-11T00:00:00,42.2,21.5,0.0,1
2,3,2301C,2023-01-06T00:00:00,IT,K01,4.8000,to_fill,500,0.0,230,2023-01-06T11:45:00,2023-01-12T00:00:00,37.5,21.5,0.0,0
3,4,2302A,2023-01-11T00:00:00,CD,K02,5.0000,to_fill,500,0.0,230,2023-01-11T11:45:00,2023-01-17T00:00:00,42.0,22.0,0.0,1
4,5,2302B,2023-01-12T00:00:00,IT,K02,5.0000,to_fill,500,0.0,230,2023-01-12T12:30:00,2023-01-17T08:30:00,38.7,21.4,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,276,2428D,2024-07-11T00:00:00,CD,K26,20.0528,to_fill,500,31.0,230,2024-07-11T09:20:00,2024-07-17T08:00:00,60.0,25.0,0.0,1
276,277,2430A,2024-07-22T00:00:00,JP,K28,20.1034,to_fill,500,30.0,230,2024-07-22T08:15:00,2024-07-29T09:10:00,54.0,21.0,0.0,1
277,278,2430B,2024-07-22T00:00:00,JP,K28,20.0612,to_fill,500,30.0,230,2024-07-22T08:15:00,2024-07-29T09:10:00,54.0,21.0,0.0,1
278,279,2430C,2024-07-22T00:00:00,JP,K28,20.0410,to_fill,500,30.0,230,2024-07-22T08:15:00,2024-07-29T09:10:00,54.0,21.0,0.0,1
